# Assignment 3 — Multi-Agent Collaboration

**Goal:** three **specialized** agents that hand work to each other through a simple coordinator:

- **SearchAgent** — retrieves factual information (DuckDuckGo)
- **AnalysisAgent** — does reasoning / computation (calculator + LLM)
- **ReportAgent** — writes a short human-readable summary (LLM)

A sequential **coordinator** runs them in order: Search -> Analysis -> Report.

In [1]:
%pip install transformers langchain langchain-community langchain-huggingface ddgs huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- LOCAL MODEL: runs offline, no token needed ---
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

llm = HuggingFacePipeline(pipeline=pipeline(
    'text2text-generation', model='google/flan-t5-base', max_new_tokens=256))
print('LLM ready: local flan-t5-base')

d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready: local flan-t5-base


C:\Users\hp\AppData\Local\Temp\ipykernel_7000\214595724.py:5: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipeline(


In [3]:
import re
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun

# --- Tool 1: web search (DuckDuckGo) ---
_ddg = DuckDuckGoSearchRun()
_ddg.api_wrapper.backend = 'html'  # real page snippets

def search_tool(query: str) -> str:
    """Search the web and return a short text snippet."""
    try:
        return _ddg.run(query)[:500]
    except Exception as e:
        return f'Search error: {e}'

# --- Tool 2: calculator (safe arithmetic only) ---
def calculator_tool(expr: str) -> str:
    """Evaluate a math expression like '4.4 * 0.05'."""
    if not re.fullmatch(r'[0-9\.\+\-\*\/\(\) ]+', expr):
        return 'Calculator error: invalid expression.'
    try:
        return str(eval(expr))
    except Exception as e:
        return f'Calculator error: {e}'

# Registry the agent will pick from
TOOLS = {'search': search_tool, 'calculator': calculator_tool}
print('Tools available:', list(TOOLS))

Tools available: ['search', 'calculator']


## The three specialized agents

Each agent is just a function with one job. They communicate by passing a shared `state` dict (the same idea you'll formalize with LangGraph's `GraphState` in Assignment 4).

In [4]:
def search_agent(state):
    print('[SearchAgent] searching:', state['query'])
    state['facts'] = search_tool(state['query'])
    return state

def analysis_agent(state):
    print('[AnalysisAgent] analyzing facts...')
    # pull the first number out of the retrieved text and compute something useful
    nums = re.findall(r'\d+(?:\.\d+)?', state['facts'])
    nums = [float(n) for n in nums if n not in ('2023','2022','2024')]
    if nums:
        value = max(nums)
        state['analysis'] = f'Largest figure found: {value}. 5% of it = {calculator_tool(str(value)+" * 0.05")}.'
    else:
        prompt = f"Briefly analyze these facts in one sentence:\n{state['facts']}"
        state['analysis'] = llm.invoke(prompt).strip()
    return state

def report_agent(state):
    print('[ReportAgent] writing summary...')
    prompt = (f"Write a short, clear summary for a human.\n"
              f"Question: {state['query']}\nFacts: {state['facts']}\nAnalysis: {state['analysis']}")
    state['report'] = llm.invoke(prompt).strip()
    return state

## The coordinator — runs the agents in sequence:

In [5]:
def coordinator(query):
    state = {'query': query}
    for agent in (search_agent, analysis_agent, report_agent):
        state = agent(state)
    return state

result = coordinator('What was the GDP of France in 2023?')
print('\n=== REPORT ===')
print(result['report'])

[SearchAgent] searching: What was the GDP of France in 2023?


d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[AnalysisAgent] analyzing facts...
[ReportAgent] writing summary...


d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\langchain_core\load\serializable.py:194: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if (k not in self.model_fields or try_neq_default(v, k, self))
d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\langchain_core\load\serializable.py:81: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = model.model_fields[key]
d:\nihal\datacamp\ML practice\ml_env\Lib\site-packages\langchain_core\load\serializable.py:194: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V


=== REPORT ===
The GDP of France in 2023 was 2026.0. 5% of it = 101.30000000000001.


## Reflection

Splitting the work into specialists keeps each agent simple and testable: the **SearchAgent** only fetches, the **AnalysisAgent** only reasons, the **ReportAgent** only writes. They collaborate through a shared `state` dict, and the **coordinator** decides the order. This is a *sequential* (rule-based) orchestration — in Assignment 4 we rebuild it in **LangGraph** and add a real decision branch.